In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

In [10]:
df = pd.read_csv("Day12_Used_Car_Preprocessing_Dataset.csv")

In [11]:
X = df.drop(columns=['Car_ID', 'Resale_Price_Lakh'])
y = df['Resale_Price_Lakh']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
num_cols = ['Year', 'Mileage_Km', 'Engine_CC', 'Power_BHP', 'Previous_Owners', 'Accidents_Reported', 'Service_Score']
ordinal_cols = ['Condition']
nominal_cols = ['Brand', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type']

In [14]:
for col in num_cols:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Clip values to bounds rather than dropping rows to retain data volume
    X_train.loc[:, col] = np.clip(X_train[col], lower_bound, upper_bound)
    X_test.loc[:, col] = np.clip(X_test[col], lower_bound, upper_bound)

/tmp/ipykernel_980/3617120022.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 65330.    76484.   112800.    66420.    98325.    52823.    77673.
  64456.    46678.    27763.    43259.   175124.25  23177.    99317.
  24399.    43014.    26207.    15147.    90330.    42304.   102042.
  82213.    33592.    92804.   109268.    22118.    42324.    80467.
  29999.    93413.    46994.   117001.   109853.    33518.    52539.
 113082.   151557.   102406.   123316.    33850.    11008.    25638.
  74819.    36445.   116022.    34749.   142937.   112940.    26028.
  83916.    73800.    76075.    64393.    61175.    70901.    99161.
  46064.    68425.     5000.    51591.    86667.    64172.    51622.
  62319.    43141.   175124.25  50293.   117048.   110191.    13208.
  83380.    81118.   102462.   110054.    99518.   102134.   170000.
  38140.    71613.    61314.    92472.    63329.   112280.   110625.
  85986.    49901.

In [15]:
condition_order = [['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']]
ordinal_encoder = OrdinalEncoder(categories=condition_order)

In [16]:
X_train_ord = pd.DataFrame(ordinal_encoder.fit_transform(X_train[ordinal_cols]),
                           columns=ordinal_cols, index=X_train.index)
X_test_ord = pd.DataFrame(ordinal_encoder.transform(X_test[ordinal_cols]),
                          columns=ordinal_cols, index=X_test.index)

In [17]:
nominal_encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

In [18]:
X_train_nom_array = nominal_encoder.fit_transform(X_train[nominal_cols])
X_test_nom_array = nominal_encoder.transform(X_test[nominal_cols])

In [19]:
nom_feature_names = nominal_encoder.get_feature_names_out(nominal_cols)
X_train_nom = pd.DataFrame(X_train_nom_array, columns=nom_feature_names, index=X_train.index)
X_test_nom = pd.DataFrame(X_test_nom_array, columns=nom_feature_names, index=X_test.index)

In [20]:
scaler = StandardScaler()

In [21]:
X_train_num = pd.DataFrame(scaler.fit_transform(X_train[num_cols]), columns=num_cols, index=X_train.index)
X_test_num = pd.DataFrame(scaler.transform(X_test[num_cols]), columns=num_cols, index=X_test.index)

In [22]:
X_train_processed = pd.concat([X_train_num, X_train_ord, X_train_nom], axis=1)
X_test_processed = pd.concat([X_test_num, X_test_ord, X_test_nom], axis=1)

In [23]:
print("Preprocessing Complete.")
print("X_train_processed shape:", X_train_processed.shape)
print("X_test_processed shape:", X_test_processed.shape)

Preprocessing Complete.
X_train_processed shape: (256, 32)
X_test_processed shape: (64, 32)


## Overview
This document outlines the data preprocessing workflow applied to the `Day12_Used_Car_Preprocessing_Dataset.csv` dataset. The goal of this pipeline is to clean, transform, and prepare the data for machine learning algorithms while strictly adhering to best practices to prevent data leakage.

## 1. Data Cleaning & Feature Separation
*   **Identifier Removal:** The `Car_ID` column was dropped entirely because it is a unique identifier and holds no predictive value for a machine learning model. Leaving it in could lead to overfitting.
*   **Target Separation:** The target variable, `Resale_Price_Lakh`, was isolated from the independent features (`X`).

## 2. Preventing Data Leakage (Train-Test Split)
*   **Decision:** The dataset was split into training (80%) and testing (20%) sets **before** applying any statistical transformations.
*   **Rationale:** If transformations (like calculating the mean for scaling or quartiles for outliers) are done on the entire dataset, information from the test set "leaks" into the training phase. By splitting first, we ensure the test set remains completely unseen, providing a reliable evaluation metric later.

## 3. Handling Outliers (IQR Method)
*   **Method:** The Interquartile Range (IQR) method was applied to all numerical columns.
*   **Decision:** Instead of dropping rows with outliers, the values were **clipped** (capped) at the lower ($Q1 - 1.5 \times IQR$) and upper ($Q3 + 1.5 \times IQR$) bounds.
*   **Rationale:** The dataset contains only 320 rows. Dropping rows would result in a significant loss of valuable training data. Clipping neutralizes the negative impact of extreme outliers on distance-based models while preserving the data volume. *Note: Bounds were calculated exclusively on the training set.*

## 4. Categorical Variable Encoding
Categorical features were split into two groups based on their inherent structure:
*   **Ordinal Encoding:** Applied to the `Condition` column. Since the categories have a natural mathematical hierarchy (`Poor` < `Fair` < `Good` < `Very Good` < `Excellent`), Ordinal Encoding was used to preserve this ranked relationship.
*   **One-Hot Encoding:** Applied to nominal variables without a specific rank (`Brand`, `Fuel_Type`, `Transmission`, `City`, `Seller_Type`).
    *   **Decision:** The `drop='first'` parameter was utilized to drop the first encoded column for each feature.
    *   **Rationale:** This prevents the "Dummy Variable Trap" (perfect multicollinearity), which can destabilize linear models.

## 5. Feature Scaling
*   **Method:** Standardization (`StandardScaler`) was applied to all continuous numerical features (`Year`, `Mileage_Km`, `Engine_CC`, `Power_BHP`, etc.).
*   **Rationale:** Features like `Mileage_Km` have drastically larger numerical scales than `Previous_Owners`. Standardization centers the features around a mean of 0 and a standard deviation of 1. This ensures that features with larger magnitudes do not disproportionately dominate the objective function during model training. *Note: The scaler was fitted only on the training data.*

## Conclusion
The dataset has been successfully preprocessed. All categorical text has been vectorized, numerical features share a uniform scale, extreme outliers have been mitigated, and the testing set remains uncontaminated. The features are now ready for model training.